# Notebook to assist in discovering checkpoints on a workspace

In [0]:
!ls ../../Users

89ece61b-2243-473f-adaa-8cc0034c2726
a541fcbd-0d96-4fcd-bb14-818b8f0e7df8
adf_user_2@mngenvmcap372892.onmicrosoft.com
admin@mngenvmcap372892.onmicrosoft.com
Groups


In [0]:
# List all catalogs; only print if they are attached to a volume
for catalog in spark.sql("SHOW CATALOGS").collect():
    cat_name = catalog[0]
    try:
        schemas = spark.sql(f"SHOW SCHEMAS IN `{cat_name}`").collect()
        for schema in schemas:
            schema_name = schema[0]
            try:
                volumes = spark.sql(f"SHOW VOLUMES IN `{cat_name}`.`{schema_name}`").collect()
                for vol in volumes:
                    vol_path = f"/Volumes/{cat_name}/{schema_name}/{vol['volume_name']}"
                    print(vol_path)
            except Exception:
                pass
    except Exception:
        pass

In [0]:
# list mounts
try:
    for mount in dbutils.fs.mounts():
        print(f"{mount.mountPoint}  →  {mount.source}")
except Exception as e:
    print(f'Error:  Likely that mounts are not allowed in workspace:\n{e}')

Error:  Likely that mounts are not allowed in workspace:
Method public com.databricks.backend.daemon.dbutils.DBUtilsCore$Result com.databricks.backend.daemon.dbutils.DBUtilsCore.mounts() is not whitelisted on class class com.databricks.backend.daemon.dbutils.DBUtilsCore


In [0]:
# list dbfs locations; note that local disk and temp have already been added to the SCAN_PATHS for search
display(dbutils.fs.ls('/'))

path,name,size,modificationTime
dbfs:/Volume/,Volume/,0,0
dbfs:/Volumes/,Volumes/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0
dbfs:/databricks-results/,databricks-results/,0,0
dbfs:/local_disk0/,local_disk0/,0,1773067574000
dbfs:/volume/,volume/,0,0
dbfs:/volumes/,volumes/,0,0


In [0]:
# Paths to scan for checkpoint directories; add locations based on where developers wrote checkpoints, such as dbfs.  Feel free to add any catalogs from the above here
SCAN_PATHS = [
  '/local_disk0/tmp',
  '/tmp',
  '/tmp/checkpoints',
  '/FileStore/tmp',
  '/FileStore/checkpoints'
]

# Max directory depth to recurse when scanning
MAX_DEPTH = 4

# Checkpoint directories contain these marker subdirectories/files
CHECKPOINT_MARKERS = {'offsets', 'commits', 'sources', 'state', 'metadata'}

print(f'Scan paths: {SCAN_PATHS}')
print(f'Max depth:  {MAX_DEPTH}')

Scan paths: ['/local_disk0/tmp', '/tmp', '/tmp/checkpoints', '/FileStore/tmp', '/FileStore/checkpoints']
Max depth:  4


In [0]:
from dataclasses import dataclass, field


@dataclass
class CheckpointInfo:
  """
  Represents a discovered checkpoint directory
  """
  path: str
  markers_found: list = field(default_factory=list)         # this is because using [] is created once and every instance will have the same value.  Instead use default_factory to create a new list for each instance
  total_size_bytes: int = 0
  file_count: int = 0

  @property
  def size_mb(self) -> float:
    return self.total_size_bytes / (1024 * 1024)

  def summary(self) -> str:
    markers = ', '.join(self.markers_found)
    return (
      f'  Path: {self.path}\n'
      f'  Markers: [{markers}]\n'
      f'  Files: {self.file_count}  |  Size: {self.size_mb:.2f} MB'
    )


def is_checkpoint_dir(path:str) -> list[str]:
  """
  Check if a DBFS directory looks like a streaming checkpoint.

  Returns the list of checkpoint marker subdirectories found,
  or an empty list if none are present.

  path:str: The path to check for checkpoint markers

  returns:
  list[str]: The list of checkpoint marker subdirectories found
  """
  try:
    entries = dbutils.fs.ls(path)
  except Exception:
    return []

  # collect the directory names
  child_names = set()
  for e in entries:
    # dbutils.fs.ls returns paths ending in / for directories
    name = e.name.rstrip('/')
    if e.isDir():
      child_names.add(name)

  found = sorted(child_names & CHECKPOINT_MARKERS)
  return found


def dir_stats(path:str) -> tuple:
  """
  Recursively compute total size and file count under a DBFS path
  Find all directories under the path and compile stats for total size and file count

  path:str: The path to scan for checkpoint directories

  returns:
  tuple: (total_size, file_count): The total size and file count
  """
  total_size = 0
  file_count = 0
  try:
    for entry in dbutils.fs.ls(path):
      if entry.isDir():
        sub_size, sub_count = dir_stats(entry.path)
        total_size = total_size + sub_size
        file_count = file_count + sub_count
      else:
        total_size = total_size + entry.size or 0
        file_count = file_count + 1
  except Exception as exc:
    print(f'  [WARN] Could not stat {path}: {exc}')
  return total_size, file_count


def scan_path_for_checkpoints(base_path:str, max_depth:int = 4) -> list:
  """
  Recursively scan a DBFS path for checkpoint directories
  Find all directories under the path and compile stats for total size and file count

  base_path:str: The path to scan for checkpoint directories
  max_depth:int: The maximum depth to recurse into the path

  returns:
  list: The list of CheckpointInfo objects
  """
  results = []

  def _recurse(path:str, depth:int) -> None:
    if depth > max_depth:
      return

    markers = is_checkpoint_dir(path)
    if markers:
      total_size, file_count = dir_stats(path)
      results.append(
        CheckpointInfo(
          path=path,
          markers_found=markers,
          total_size_bytes=total_size,
          file_count=file_count,
        )
      )
      return  # don't recurse into a checkpoint dir's children

    try:
      entries = dbutils.fs.ls(path)
    except Exception:
      return

    for entry in entries:
      if entry.isDir():
        _recurse(entry.path, depth + 1)

  _recurse(base_path, 0)
  return results


print('Helper functions defined.')

Helper functions defined.


# Discover Checkpoints

In [0]:
all_checkpoints = []

for scan_path in SCAN_PATHS:
  print(f'Scanning {scan_path} ...')
  try:
    found = scan_path_for_checkpoints(scan_path, max_depth=MAX_DEPTH)
    all_checkpoints.extend(found)
    print(f'  Found {len(found)} checkpoint(s) under {scan_path}')
  except Exception as exc:
    print(f'  Skipping {scan_path} — {exc}')

print(f'\n{'=' * 60}')
print(f'Checkpoint Discovery Report')
print(f'{'=' * 60}')

if not all_checkpoints:
  print('\nNo checkpoint directories found.')
  print(
    'Note: Default temp checkpoints on local_disk0 are ephemeral and may\n'
    '     not survive cluster restarts. Try adding custom paths to\n'
    '     SCAN_PATHS if your jobs wrote checkpoints elsewhere.'
  )
else:
  print(f'\nFound {len(all_checkpoints)} checkpoint(s):\n')
  for i, cp in enumerate(all_checkpoints, 1):
    print(f'[{i}]')
    print(cp.summary())
    print()

  total_mb = sum(cp.size_mb for cp in all_checkpoints)
  print(f'Total size: {total_mb:.2f} MB across {len(all_checkpoints)} checkpoint(s)')

Scanning /local_disk0/tmp ...
  Found 0 checkpoint(s) under /local_disk0/tmp
Scanning /tmp ...
  Found 0 checkpoint(s) under /tmp
Scanning /tmp/checkpoints ...
  Found 0 checkpoint(s) under /tmp/checkpoints
Scanning /FileStore/tmp ...
  Found 0 checkpoint(s) under /FileStore/tmp
Scanning /FileStore/checkpoints ...
  Found 0 checkpoint(s) under /FileStore/checkpoints

Checkpoint Discovery Report

No checkpoint directories found.
Note: Default temp checkpoints on local_disk0 are ephemeral and may
     not survive cluster restarts. Try adding custom paths to
     SCAN_PATHS if your jobs wrote checkpoints elsewhere.


# Display Results as a Table

In [0]:
if all_checkpoints:
  from pyspark.sql import Row

  rows = [
    Row(
      path=cp.path,
      markers=", ".join(cp.markers_found),
      file_count=cp.file_count,
      size_mb=round(cp.size_mb, 2),
    )
    for cp in all_checkpoints
  ]
  df_checkpoints = spark.createDataFrame(rows)
  display(df_checkpoints)
else:
  print("No checkpoints to display.")

No checkpoints to display.


# Migrate Checkpoint Files to ADLS

In [0]:
def migrate_checkpoint_files_to_adls(inp_path:str, land_path:str) ->None:
    """
    inp_path:str: Full file path of location on Databricks
    land_path:str: Full file path to land location on ADLS
    """
    try:
        dbutils.fs.cp(inp_path, land_path, recurse=True)
        print(f'Successfully copied {inp_path} to {land_path}')
    except Exception as e:
        print(f'Error moving checkpoint files to ADLS\nCheck the input path:{inp_path} and output path:{land_path}\nFull Error: {e}')
        return


In [0]:
# move directories to ADLS
CONTAINER = 'healthdata'
STORAGE_ACCOUNT = 'healthsaadlsrheus'
CHECKPOINT_PATH = 'checkpoints/test_migration'
target_adls_path = f'abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CHECKPOINT_PATH}'

print(f'Target ADLS path: {target_adls_path}')
print(f'Checkpoints to migrate: {len(all_checkpoints)}\n')

for _ in all_checkpoints:
    migrate_checkpoint_files_to_adls(_.path, target_adls_path)